# grid_py Tutorial 1: CairoRenderer Basics

This notebook demonstrates the core grid_py drawing primitives using the **CairoRenderer** backend, which outputs static PNG/PDF/SVG.

grid_py is a Python port of R's `grid` graphics system — the low-level engine behind ggplot2.

In [ ]:
import numpy as np
from IPython.display import display, Image

from grid_py import (
    CairoRenderer, Gpar, Unit, Viewport, GridLayout,
    get_state, grid_draw, grid_newpage,
    push_viewport, pop_viewport,
    rect_grob, circle_grob, text_grob, lines_grob,
    points_grob, polygon_grob, segments_grob,
    roundrect_grob, path_grob,
)

def show(renderer):
    """Display a CairoRenderer's output inline."""
    display(Image(data=renderer.to_png_bytes()))

## 1. Basic Shapes

All coordinates are in **NPC** (Normalised Parent Coordinates): `(0,0)` = bottom-left, `(1,1)` = top-right.

In [ ]:
r = CairoRenderer(width=6, height=4, dpi=150, bg='white')
state = get_state()
state.init_device(r)

# Rectangle
grid_draw(rect_grob(x=0.2, y=0.5, width=0.25, height=0.6,
                     gp=Gpar(fill='#3498db', col='#2c3e50', lwd=2)))

# Circle
grid_draw(circle_grob(x=0.5, y=0.5, r=0.15,
                       gp=Gpar(fill='#e74c3c', col='#c0392b', lwd=2)))

# Rounded rectangle
grid_draw(roundrect_grob(x=0.8, y=0.5, width=0.25, height=0.6, r=Unit(0.03, 'npc'),
                          gp=Gpar(fill='#2ecc71', col='#27ae60', lwd=2)))

# Labels
grid_draw(text_grob(label='rect', x=0.2, y=0.1, gp=Gpar(fontsize=11, col='#2c3e50')))
grid_draw(text_grob(label='circle', x=0.5, y=0.1, gp=Gpar(fontsize=11, col='#2c3e50')))
grid_draw(text_grob(label='roundrect', x=0.8, y=0.1, gp=Gpar(fontsize=11, col='#2c3e50')))

# Title
grid_draw(text_grob(label='Basic Shapes', x=0.5, y=0.95,
                     gp=Gpar(fontsize=16, fontface='bold', col='#2c3e50')))

show(r)

## 2. Lines, Points, and Polygons

In [ ]:
r = CairoRenderer(width=6, height=4, dpi=150, bg='#fafafa')
state.init_device(r)

# Scatter points with different pch shapes
np.random.seed(42)
n = 30
x = np.random.rand(n)
y = np.random.rand(n) * 0.7 + 0.15
cols = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6'] * 6
grid_draw(points_grob(x=x, y=y, pch=19,
                       gp=Gpar(col=cols[:n], fill=cols[:n], fontsize=8)))

# Line through data
order = np.argsort(x)
grid_draw(lines_grob(x=x[order], y=y[order],
                      gp=Gpar(col='#2c3e50', lwd=1.5, lty='dashed')))

# Polygon (triangle)
grid_draw(polygon_grob(x=[0.85, 0.95, 0.9], y=[0.05, 0.05, 0.15],
                        gp=Gpar(fill='#f39c12', col='#e67e22', lwd=2)))

# Segments (axis-like ticks)
tick_x = np.linspace(0.05, 0.95, 10)
grid_draw(segments_grob(
    x0=tick_x, y0=np.full(10, 0.1),
    x1=tick_x, y1=np.full(10, 0.12),
    gp=Gpar(col='grey40')))

grid_draw(text_grob(label='Scatter + Lines + Polygon', x=0.5, y=0.95,
                     gp=Gpar(fontsize=14, fontface='bold')))

show(r)

## 3. Viewports and Layouts

Viewports are the heart of grid — they define nested coordinate systems. A `GridLayout` divides space into rows and columns.

In [ ]:
r = CairoRenderer(width=7, height=5, dpi=150, bg='white')
state.init_device(r)

# Title
grid_draw(text_grob(label='2x2 Grid Layout', x=0.5, y=0.97,
                     gp=Gpar(fontsize=16, fontface='bold')))

# Create a 2x2 layout
layout = GridLayout(
    nrow=2, ncol=2,
    widths=Unit([1, 2], 'null'),    # 2nd col is twice as wide
    heights=Unit([1, 1], 'null'),
)

# Push the layout viewport (inset slightly)
main_vp = Viewport(name='main', x=0.5, y=0.47, width=0.9, height=0.85, layout=layout)
push_viewport(main_vp, recording=False)
r.push_viewport(main_vp)

# Draw in each cell
cell_colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
cell_labels = ['Cell (1,1)', 'Cell (1,2)', 'Cell (2,1)', 'Cell (2,2)']

for row in [1, 2]:
    for col in [1, 2]:
        idx = (row - 1) * 2 + (col - 1)
        vp = Viewport(name=f'cell_{row}_{col}',
                       layout_pos_row=row, layout_pos_col=col)
        push_viewport(vp, recording=False)
        r.push_viewport(vp)

        # Background
        grid_draw(rect_grob(x=0.5, y=0.5, width=0.95, height=0.92,
                             gp=Gpar(fill=cell_colors[idx] + '30',
                                     col=cell_colors[idx], lwd=2)))
        # Label
        grid_draw(text_grob(label=cell_labels[idx], x=0.5, y=0.5,
                             gp=Gpar(fontsize=14, col=cell_colors[idx],
                                     fontface='bold')))

        pop_viewport(1, recording=False)
        r.pop_viewport()

pop_viewport(1, recording=False)
r.pop_viewport()

show(r)

## 4. Gradients and Transparency

In [ ]:
from grid_py._patterns import LinearGradient, RadialGradient

r = CairoRenderer(width=6, height=4, dpi=150, bg='#1a1a2e')
state.init_device(r)

# Linear gradient rectangle
lg = LinearGradient(
    colours=['#e94560', '#0f3460'],
    stops=[0.0, 1.0],
    x1=Unit(0, 'npc'), y1=Unit(0, 'npc'),
    x2=Unit(1, 'npc'), y2=Unit(1, 'npc'),
)
grid_draw(rect_grob(x=0.3, y=0.5, width=0.4, height=0.7,
                     gp=Gpar(fill=lg, col='#e94560', lwd=2)))

# Semi-transparent overlapping circles
for cx, color in [(0.65, '#e94560'), (0.75, '#0f3460'), (0.85, '#16213e')]:
    grid_draw(circle_grob(x=cx, y=0.5, r=0.15,
                           gp=Gpar(fill=color, alpha=0.6, col='white', lwd=1)))

grid_draw(text_grob(label='Gradients & Alpha', x=0.5, y=0.95,
                     gp=Gpar(fontsize=16, fontface='bold', col='white')))

show(r)

## 5. Output Formats

CairoRenderer supports PNG, PDF, and SVG output.

In [ ]:
# Save as PNG
r.write_to_png('output_demo.png')
print('Saved output_demo.png')

# Save as PDF
r_pdf = CairoRenderer(width=6, height=4, dpi=72,
                        surface_type='pdf', filename='output_demo.pdf')
r_pdf.draw_rect(0.5, 0.5, 0.8, 0.6, gp=Gpar(fill='#3498db', col='black'))
r_pdf.draw_text(0.5, 0.5, 'PDF Output', gp=Gpar(fontsize=20, col='white'))
r_pdf.finish()
print('Saved output_demo.pdf')

# Save as SVG
r_svg = CairoRenderer(width=6, height=4, dpi=72,
                        surface_type='svg', filename='output_demo.svg')
r_svg.draw_circle(0.5, 0.5, 0.3, gp=Gpar(fill='#e74c3c'))
r_svg.draw_text(0.5, 0.5, 'SVG Output', gp=Gpar(fontsize=20, col='white'))
r_svg.finish()
print('Saved output_demo.svg')